# z303 – Optuna + MLflow
**Grupo 3: Banegas - Marín - Mengoni - Rey**

Búsqueda de hiperparámetros con Optuna (TPE + MedianPruner) y tracking en MLflow.
Lee `dataset_fe.parquet` de z302. Guarda mejor combinación en JSON + SQLite → GCS.

**Para ver los dashboards:**
```bash
# MLflow (en una terminal de la VM)
mlflow ui --backend-store-uri file:///home/ds/buckets/b1/mlruns --port 5000
# luego: ssh -L 5000:localhost:5000 tu-vm

# Optuna Dashboard (en otra terminal)
optuna-dashboard sqlite:////home/ds/optuna_B_zero_full.db --port 8080
# luego: ssh -L 8080:localhost:8080 tu-vm
```

In [ ]:
import yaml, json, time
from pathlib import Path
import numpy as np
import duckdb
import lightgbm as lgb
import optuna
import mlflow
from sklearn.model_selection import KFold
from google.cloud import storage as gcs

optuna.logging.set_verbosity(optuna.logging.WARNING)

with open('../pipe_py/config.yaml') as f:
    CFG = yaml.safe_load(f)

# --- palancas ---
CFG['optuna']['n_trials']   = 80
CFG['optuna']['n_folds']    = 5
CFG['optuna']['metrica']    = 'wape'
CFG['fe']['tipo_target']    = 'nivel'   # nivel | delta
# ----------------

pr   = CFG['preproc']
modo = f"{pr['group_mode']}_{pr['missing_strategy']}_{pr['densify_strategy']}"
print(f'Modo: {modo}')

In [ ]:
# Cargar datos con DuckDB
ruta_fe = Path(CFG['paths']['fe_out']) / 'dataset_fe.parquet'
con = duckdb.connect()
con.execute(f"CREATE TABLE ds AS SELECT * FROM read_parquet('{ruta_fe}')")

excluir = set(CFG['fe']['cols_excluir_leakage'] + ['target', 'Agrupacion_ID', 'B0', 'B1'])
feature_cols = [r[0] for r in con.execute('DESCRIBE ds').fetchall() if r[0] not in excluir]

datos = con.execute(f"SELECT {', '.join(feature_cols + ['target'])} FROM ds").fetchnumpy()
X = np.column_stack([datos[c] for c in feature_cols])
y = datos['target']
print(f'X: {X.shape}  |  features: {len(feature_cols)}')

In [ ]:
# WAPE
def wape(y_true, y_pred):
    d = np.abs(y_true).sum()
    return np.abs(y_true - y_pred).sum() / d if d > 0 else 0.0

# Objective
opt_cfg = CFG['optuna']
space   = CFG['lgbm_space']

def objective(trial):
    params = {
        'objective':         opt_cfg['objective_lgbm'],
        'num_leaves':        trial.suggest_int('num_leaves', *space['num_leaves']),
        'learning_rate':     trial.suggest_float('learning_rate', *space['learning_rate'], log=True),
        'n_estimators':      trial.suggest_int('n_estimators', *space['n_estimators']),
        'min_child_samples': trial.suggest_int('min_child_samples', *space['min_child_samples']),
        'subsample':         trial.suggest_float('subsample', *space['subsample']),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', *space['colsample_bytree']),
        'reg_alpha':         trial.suggest_float('reg_alpha', *space['reg_alpha']),
        'reg_lambda':        trial.suggest_float('reg_lambda', *space['reg_lambda']),
        'verbosity': -1, 'n_jobs': -1, 'random_state': opt_cfg['semilla'],
    }
    kf = KFold(n_splits=opt_cfg['n_folds'], shuffle=True, random_state=opt_cfg['semilla'])
    wapes = []
    for fold, (itr, iva) in enumerate(kf.split(X)):
        m = lgb.LGBMRegressor(**params)
        m.fit(X[itr], y[itr], eval_set=[(X[iva], y[iva])],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
        w = wape(y[iva], m.predict(X[iva]))
        wapes.append(w)
        trial.report(w, fold)
        if trial.should_prune(): raise optuna.TrialPruned()
    return float(np.mean(wapes))

In [ ]:
# MLflow setup
mlflow.set_tracking_uri(CFG['paths']['mlflow_uri'])
mlflow.set_experiment(opt_cfg['experiment_name'])

# Optuna study
sqlite_path = opt_cfg['sqlite_path'].replace('{MODO}', modo)
sampler = optuna.samplers.TPESampler(seed=opt_cfg['semilla'])
pruner  = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1)
study   = optuna.create_study(
    study_name=f'labo3_{modo}',
    direction='minimize',
    sampler=sampler, pruner=pruner,
    storage=f'sqlite:///{sqlite_path}',
    load_if_exists=True,
)
print(f'SQLite: {sqlite_path}')

In [ ]:
# Optimizar
with mlflow.start_run(run_name=f'optuna_{modo}') as run:
    mlflow.log_params({'modo': modo, 'n_trials': opt_cfg['n_trials'],
                       'tipo_target': CFG['fe']['tipo_target']})
    t0 = time.time()
    study.optimize(objective, n_trials=opt_cfg['n_trials'], show_progress_bar=True)
    elapsed = time.time() - t0
    best = study.best_trial
    mlflow.log_metric('best_wape', best.value)
    mlflow.log_metric('elapsed_seg', elapsed)
    run_id = run.info.run_id

print(f'\nMejor trial #{best.number}  WAPE={best.value:.4f}  ({elapsed:.0f}s)')
print(f'MLflow run_id: {run_id}')

In [ ]:
# Visualización trials con DuckDB
trials_data = [
    {'trial': t.number, 'wape': t.value, 'estado': t.state.name, **t.params}
    for t in study.trials if t.value is not None
]
con2 = duckdb.connect()
import json as _json
# crear tabla
con2.execute("CREATE TABLE trials AS SELECT * FROM trials_data")
con2.execute("""
    SELECT trial, wape,
           MIN(wape) OVER (ORDER BY trial ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS mejor_hasta_ahora
    FROM trials ORDER BY trial
""").df().plot(x='trial', y=['wape','mejor_hasta_ahora'], figsize=(10,4), title='WAPE por trial')

In [ ]:
# Importancia de hiperparámetros (fANOVA)
try:
    importancias = optuna.importance.get_param_importances(study)
    print('Importancia de hiperparámetros:')
    for k, v in importancias.items():
        print(f'  {k}: {v:.3f}')
except Exception as e:
    print(f'fANOVA no disponible: {e}')

In [ ]:
# Guardar resultado y subir a GCS
resultado = {
    'modo': modo, 'tipo_target': CFG['fe']['tipo_target'],
    'metrica': opt_cfg['metrica'], 'best_wape': best.value,
    'best_params': best.params, 'features': feature_cols,
    'n_trials': len(study.trials), 'mlflow_run_id': run_id,
}
ruta_json = Path(CFG['paths']['optuna_out']) / f'z303_hiper_{modo}.json'
ruta_json.parent.mkdir(parents=True, exist_ok=True)
with open(ruta_json, 'w') as f:
    json.dump(resultado, f, indent=2)

g = CFG['gcs']
client = gcs.Client()
bucket = client.bucket(g['bucket'])
for ruta, blob_name in [(ruta_json, f"z303_hiper_{modo}.json"),
                         (Path(sqlite_path), f"z303_optuna_{modo}.db")]:
    b = bucket.blob(f"{g['prefix_optuna']}/{blob_name}")
    b.upload_from_filename(str(ruta))
    print(f"Subido → gs://{g['bucket']}/{g['prefix_optuna']}/{blob_name}")

print('\nMejores hiperparámetros:')
for k, v in best.params.items():
    print(f'  {k}: {v}')